# IOAI — 2025 National Byzantine Notation (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train_data.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-byzantine.zip', '/tmp/byz.zip')
    with zipfile.ZipFile('/tmp/byz.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train_data.csv' if os.path.exists('data/train_data.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Byzantine Notation — 멀티-neume 시퀀스 예측 (모범답안)
**핵심: 분포 정렬(distribution alignment)** — 학습용 단일 neume 도 테스트와 **동일한 `extract_patches`**
(이진화→바운딩박스 크롭→48×48)를 거쳐 전처리해, 분류기가 실제로 보게 될 패치와 같은 분포로 학습합니다.
여기에 **데이터 증강 + 소형 CNN**(RoAI reference 방식)을 더합니다. 채점 = held-out 합성 strip **seq_acc**.
(원대회 원본 reference 전체 파이프라인은 **모범답안2** 참고.)

In [ ]:
import cv2 as cv, numpy as np, pandas as pd
from PIL import Image
def remove_background(img):
    _,img=cv.threshold(img,120,255,cv.THRESH_BINARY)
    blur=cv.GaussianBlur(img,(3,3),cv.BORDER_DEFAULT)
    return cv.adaptiveThreshold(blur,255,cv.ADAPTIVE_THRESH_MEAN_C,cv.THRESH_BINARY,15,10)
def extract_patches(img,min_area=100,pad=2):
    img=remove_background(img)
    thr=cv.adaptiveThreshold(img,255,cv.ADAPTIVE_THRESH_MEAN_C,cv.THRESH_BINARY_INV,15,10)
    cnts,_=cv.findContours(thr,cv.RETR_EXTERNAL,cv.CHAIN_APPROX_SIMPLE)
    ps=[]
    for c in cnts:
        x,y,w,h=cv.boundingRect(c)
        if w*h<min_area: continue
        xs,ys=max(x-pad,0),max(y-pad,0); xe,ye=min(x+w+pad,img.shape[1]),min(y+h+pad,img.shape[0])
        ps.append((xs,img[ys:ye,xs:xe]))
    return [p for _,p in sorted(ps,key=lambda z:z[0])]
def encode_prediction(labels):
    res=[]
    for i,p in enumerate(labels):
        if i==0: res.append(0 if p in ("A","B") else int(p))
        else: res.append(res[-1] if p in ("A","B") else res[-1]+int(p))
    return res

In [ ]:
df=pd.read_csv('data/train_data.csv')
is_val=df.index%5==0          # held-out 단일 neume → 합성 strip(eval_local) 채점 전용(학습 제외)
trn=df[~is_val].reset_index(drop=True)
labels=sorted(df['Effect'].astype(str).unique()); L2I={l:i for i,l in enumerate(labels)}; I2L={i:l for l,i in L2I.items()}
print('classifier train neumes:',len(trn),'| classes',labels)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms
dev='cuda' if torch.cuda.is_available() else 'cpu'
def proc(path):                      # ★ 학습 neume 를 테스트와 같은 extract_patches 로 정렬
    img=cv.imread('data/'+path,cv.IMREAD_GRAYSCALE); pts=extract_patches(img)
    p=max(pts,key=lambda z:z.size) if pts else remove_background(img)
    return np.array(Image.fromarray(p).resize((48,48)),dtype=np.float32)/255.
Xtr=np.stack([proc(p) for p in trn['Path']]); ytr=np.array([L2I[str(e)] for e in trn['Effect']]); yt=torch.tensor(ytr)
norm=transforms.Compose([transforms.ToTensor(),transforms.Normalize([0.5],[0.5])]); ra=transforms.RandAugment(2,7)
def batch(X,idx,train):
    o=[]
    for i in idx:
        im=Image.fromarray((X[i]*255).astype('uint8'))
        if train: im=ra(im)
        o.append(norm(im))
    return torch.stack(o)
class CNN(nn.Module):
    def __init__(s,n):
        super().__init__(); s.c1=nn.Conv2d(1,8,3,padding=1); s.b1=nn.BatchNorm2d(8)
        s.c2=nn.Conv2d(8,16,3,padding=1); s.b2=nn.BatchNorm2d(16); s.p=nn.MaxPool2d(2,2)
        s.f1=nn.LazyLinear(128); s.dp=nn.Dropout(0.5); s.f2=nn.Linear(128,n)
    def forward(s,x):
        x=s.p(F.relu(s.b1(s.c1(x)))); x=s.p(F.relu(s.b2(s.c2(x))))
        x=x.view(x.size(0),-1); x=s.dp(F.relu(s.f1(x))); return s.f2(x)
torch.manual_seed(0); m=CNN(len(labels)).to(dev)
opt=torch.optim.AdamW(m.parameters(),1e-3,weight_decay=1e-4); crit=nn.CrossEntropyLoss()
for ep in range(60):
    m.train(); perm=np.random.permutation(len(Xtr))
    for sidx in range(0,len(perm),64):
        idx=perm[sidx:sidx+64]; opt.zero_grad(); crit(m(batch(Xtr,idx,True).to(dev)),yt[idx].to(dev)).backward(); opt.step()
m.eval()
def predict_patch(P):
    with torch.no_grad():
        xb=torch.stack([norm(Image.fromarray((p*255).astype('uint8'))) for p in P]).to(dev)
        return m(xb).argmax(1).cpu().numpy()

In [ ]:
# 멀티-neume strip → 패치 분리 → 분류 → 누적 시퀀스 인코딩 → 제출(원대회 형식: subtaskID,datapointID,answer)
ev=pd.read_csv('data/eval_local.csv'); rows=[]
for _,r in ev.iterrows():
    img=cv.imread('data/'+r['datapointID'],cv.IMREAD_GRAYSCALE); patches=extract_patches(img)
    if patches:
        P=np.stack([np.array(Image.fromarray(p).resize((48,48)),dtype=np.float32)/255. for p in patches])
        ans='|'.join(map(str,encode_prediction([I2L[i] for i in predict_patch(P)])))
    else: ans=''
    rows.append({'subtaskID':1,'datapointID':r['datapointID'],'answer':ans})
pd.DataFrame(rows).to_csv('submission.csv',index=False); print('wrote submission.csv',len(rows))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)